## Disambiguazione
Questo notebook raccoglie gli effort per la disambiguazione degli autori. 
1. Aggiunta id partendo dal dizionario che mappa autore-> id
2. Algoritmo di ulteriore disambiguazione

In [2]:
import json
import re
import os
import math

import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict

import requests
import json
import time
import os
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [9]:
dict_autori_id = "/Users/laviniarotellini/Desktop/Tesi/Step 2: Disambiguazione dei nomi/checkpoint_authors_it.json"
with open(dict_autori_id, 'r') as outfile_1:
    checkpoint_it = json.load(outfile_1)

In [ ]:
# Spostato in prove varie per fare ordine-> è servito solo per la prima passata
#dict_autori_isti_id = "/Users/laviniarotellini/Desktop/Tesi/Step 2: Disambiguazione dei nomi/checkpoint.json 12-24-23-379.json"
#with open(dict_autori_isti_id, 'r') as outfile_2:
#    checkpoint = json.load(outfile_2)

## 1 

In [10]:
# Estrai i dizionari autori
authors_it = checkpoint_it.get("authors", {})      
#authors_cache = checkpoint.get("author_cache", {})

# Carica il file principale da modificare
with open("papers_merged.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
# Prima funzione da applicare a papers_merged per modificarne la struttura così da avere authors-> sottodizionario con name, affiliation, id
# Si implementa anche una prima aggiunta di ID, ma è un primo tentativo con i pochi dati scaricati fino ad ora
'''
def get_author_id_on_papers_merged(name, affiliation):
    """
    Cerca l'id nei due dizionari con questa priorità:
    1. authors_it con chiave "Nome||Affiliazione"
    2. authors_it con solo il nome
    3. author_cache con chiave "Nome||Affiliazione"
    4. author_cache con solo il nome (primo match)
    """
    key_with_aff = f"{name}||{affiliation}"

    # 1. Cerca in authors_it con affiliazione
    if key_with_aff in authors_it:
        return authors_it[key_with_aff]

    # 2. Cerca in authors_it solo per nome
    if name in authors_it:
        return authors_it[name]

    # 3. Cerca in author_cache con affiliazione
    if key_with_aff in authors_cache:
        return authors_cache[key_with_aff]

    # 4. Cerca in author_cache solo per nome (prende il primo match)
    for key, url in authors_cache.items():
        if key.startswith(f"{name}||"):
            return url

    return None

'''

In [ ]:
# Aggiorna il dizionario
'''
for title, paper in data.items():
    new_authors = []
    for author_entry in paper["authors"]:
        # Salta se è già un dict con 'id'
        if isinstance(author_entry, dict) and "id" in author_entry:
            new_authors.append(author_entry)
            continue

        # Estrai nome e affiliazione dalla lista [nome, affiliazione]
        if isinstance(author_entry, list):
            name = author_entry[0]
            affiliation = author_entry[1] if len(author_entry) > 1 else ""
        else:
            name = str(author_entry)
            affiliation = ""

        author_id = get_author_id_on_papers_merged(name, affiliation)

        new_authors.append({
            "name": name,
            "affiliation": affiliation,
            "id": author_id  # None → null in JSON
        })

    paper["authors"] = new_authors

# Salva il risultato
with open("before_my_disambiguation.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Fatto!")

# Statistiche
total = sum(len(p["authors"]) for p in data.values())
found = sum(1 for p in data.values() for a in p["authors"] if a["id"] is not None)
missing = total - found
print(f"Autori con ID trovato: {found}/{total}")
print(f"Autori senza ID (null): {missing}/{total}")
'''

Fatto!
Autori con ID trovato: 187712/448487
Autori senza ID (null): 260775/448487


In [12]:
## RUNNARE DA QUA PER AGGIORNAMENTI
with open("before_my_disambiguation.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [14]:
# Nuova funzione per lavorare sul nuovo dizionario before_my_disambiguation che ha una struttura migliorata
# Continuiamo ad aggiungere i dati direttamente qua prendendoli da checkpoint_authors_it e basta
def get_author_id(name, affiliation):
    """
    Cerca l'id solo in authors_it (checkpoint_authors_it.json).
    Prima con "Nome||Affiliazione", poi solo con "Nome".
    """
    key_with_aff = f"{name}||{affiliation}"

    if key_with_aff in authors_it:
        return authors_it[key_with_aff]

    if name in authors_it:
        return authors_it[name]

    return None

# Aggiorna il dizionario
for title, paper in data.items():
    for author_entry in paper["authors"]:
        # Salta se ha già un id
        if author_entry.get("id") is not None:
            continue

        name = author_entry["name"]
        affiliation = author_entry.get("affiliation", "")

        author_id = get_author_id(name, affiliation)

        if author_id is not None:
            author_entry["id"] = author_id
        # Se non trovato, lascia id = null com'è

In [15]:
total = sum(len(p["authors"]) for p in data.values())
found = sum(1 for p in data.values() for a in p["authors"] if a["id"] is not None)
missing = total - found
print(f"Autori con ID trovato: {found}/{total}")
print(f"Autori senza ID (null): {missing}/{total}")

Autori con ID trovato: 379231/448487
Autori senza ID (null): 69256/448487


In [ ]:
# Creiamo una lista con i nominativi rimanenti e proviamo a cercare solo quelli
not_found = []
for paper, info in data.items():
    authors = info.get('authors')
    for i in authors:
        if i['id'] == None:
            not_found.append(i)

# Codice per lavorare direttamente sugli assenti

OPENALEX_BASE_URL = "https://api.openalex.org/authors"
CHECKPOINT_FILE = "checkpoint.json"
OUTPUT_FILE = "updated_list.json"

EMAIL = "l.rotellini@studenti.unipi.it"
API_KEY = "4dpyiAsm6AXwpVBrOtVKwY"

if not API_KEY:
    raise ValueError("OPENALEX_API_KEY non trovata nelle variabili d'ambiente.")

# Retry in caso di errore

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)

session.mount("https://", HTTPAdapter(max_retries=retries))

session.headers.update({
    "Authorization": f"Bearer {API_KEY}"
})

### QUERY ###

def query_openalex_author(name, affiliation=None):
    params = {
        "filter": f"display_name.search:{name}",
        "per-page": 5,
        "mailto": EMAIL
    }

    try:
        response = session.get(OPENALEX_BASE_URL, params=params, timeout=30)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"Errore API per {name}: {e}")
        return None

    results = response.json().get("results", [])

    if not results:
        return None

    # Matching con affiliation
    for author in results:
        if affiliation:
            institutions_data = author.get("last_known_institutions") or []

            institutions = [
                    inst.get("institution", {}).get("display_name")
                    for inst in institutions_data
                    if inst.get("institution")
                ]

            if any(affiliation.lower() in inst.lower() for inst in institutions):
                return author["id"]

    # fallback
    return results[0]["id"]


### SALVATAGGIO CHECKPOINT ###

def save_checkpoint(index):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"last_index": index}, f)


def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f).get("last_index", 0)
    return 0


### CORPO PRINCIPALE ###

def enrich_authors(author_list):
    start_index = load_checkpoint()
    
    # Cache dei nomi già processati, con affiliazione
    processed_cache = set()

    for i in range(start_index, len(author_list)):
        author = author_list[i]

        # normalizzazione nome + affiliazione per la cache
        key = (
            (author["name"] or "").strip().lower(),
            (author.get("affiliation") or "").strip().lower()
        )

        # se già processato con stessa affiliazione → skip
        if key in processed_cache:
            print(f"Skipping duplicate: {author['name']} ({author.get('affiliation')})")
            continue

        # aggiungi alla cache
        processed_cache.add(key)

        # stampa per debug
        print(f"[{i+1}/{len(author_list)}] Processing {author['name']} ({author.get('affiliation')})")

        # chiamata OpenAlex
        openalex_id = query_openalex_author(
            author["name"],
            author.get("affiliation")
        )

        author["id"] = openalex_id

        # salva checkpoint
        save_checkpoint(i + 1)

        # salva output progressivo
        with open(OUTPUT_FILE, "w") as f:
            json.dump(author_list, f, indent=2)

        # pausa per rate limit
        time.sleep(0.2)

    print("Completato.")
    return author_list

enrich_authors(not_found)

In [16]:
# Lista di dizionari con gli id degli autori mancanti
with open("updated_list.json", "r", encoding="utf-8") as f:
    updated_list = json.load(f)

In [17]:
# Riuniamo not_found con before_my_disambiguation
def merge_dictionaries(dict_1, dict_2):
    # Converti dict_2 in un dizionario {name: id} per ricerca rapida
    lookup = {entry['name']: entry['id'] for entry in dict_2}
    
    for paper, info in dict_1.items():
        authors = info.get('authors', [])
        for author in authors:
            if author['id'] is None:
                if author['name'] in lookup:
                    author['id'] = lookup[author['name']]
    
    return dict_1

In [18]:
data = merge_dictionaries(data, updated_list)

In [19]:
total = sum(len(p["authors"]) for p in data.values())
found = sum(1 for p in data.values() for a in p["authors"] if a["id"] is not None)
missing = total - found
print(f"Autori con ID trovato: {found}/{total}")
print(f"Autori senza ID (null): {missing}/{total}")

Autori con ID trovato: 380187/448487
Autori senza ID (null): 68300/448487


## 2. Creazione dizionari che riunisce sotto stesso ID nomi diversi

In [20]:
def build_author_dict(papers):
    author_dict = {}
    for paper, info in papers.items():
        year = info.get('publication_year')
        paper_authors = info.get('authors', [])
        macroareas = info.get('macroareas', [])

        for author in paper_authors:
            author_id = author['id']
            if author_id is None:
                continue
            name = author['name']
            affiliation = author.get('affiliation')

            if author_id not in author_dict:
                author_dict[author_id] = {}
            if name not in author_dict[author_id]:
                author_dict[author_id][name] = {
                    'affiliations_by_year': {},
                    'papers': {}
                }

            entry = author_dict[author_id][name]

            if year not in entry['affiliations_by_year']:
                entry['affiliations_by_year'][year] = set()
            if affiliation:
                entry['affiliations_by_year'][year].add(affiliation)

            if paper not in entry['papers']:
                entry['papers'][paper] = {
                    'year': year,
                    'macroareas': macroareas,
                    'collaborators': []
                }

            # Collaboratori univoci per id, escludendo l'autore stesso
            seen_ids = set()
            unique_collaborators = []
            for a in paper_authors:
                if a['id'] == author_id:
                    continue
                if a['id'] not in seen_ids:
                    seen_ids.add(a['id'])
                    unique_collaborators.append({'name': a['name'], 'id': a['id']})

            entry['papers'][paper]['collaborators'] = unique_collaborators

    for author_id, names in author_dict.items():
        for name, data in names.items():
            data['affiliations_by_year'] = {
                year: list(affs) for year, affs in data['affiliations_by_year'].items()
            }

    return author_dict

In [21]:
author_dict = build_author_dict(data)

In [22]:
len(author_dict)

86153

In [23]:
with open('final_pre_disambiguation.json', 'w') as f:
    json.dump(author_dict, f)

## 3. Applicazione name disambiguation
Preso da Miranda-González et al. (2020)


In [39]:
import re
import numpy as np
from collections import defaultdict

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

from rapidfuzz import fuzz

In [25]:
def normalize_author_name(name):

    name = name.lower()
    name = re.sub(r"[^\w\s]", "", name)

    tokens = name.split()

    if len(tokens) == 0:
        return ""

    surname = tokens[-1]
    initials = "".join([t[0] for t in tokens[:-1]])

    return surname + "_" + initials

In [26]:
def normalize_affiliations(aff_list):

    stopwords = {
        "university","universita","università","dept",
        "department","school","faculty","of","the"
    }

    tokens = []

    for aff in aff_list:

        aff = aff.lower()
        aff = re.sub(r"[^\w\s]", " ", aff)

        for w in aff.split():
            if w not in stopwords:
                tokens.append(w)

    return set(tokens)

In [27]:
def blocking_key(name):

    name = name.lower()
    name = re.sub(r"[^\w\s]", "", name)

    tokens = name.split()

    if len(tokens) == 0:
        return ""

    surname = tokens[-1]
    first_initial = tokens[0][0]

    return surname + "_" + first_initial

In [28]:
def blocking_key(name):

    name = name.lower()
    name = re.sub(r"[^\w\s]", "", name)

    tokens = name.split()

    if len(tokens) == 0:
        return ""

    surname = tokens[-1]
    first_initial = tokens[0][0]

    return surname + "_" + first_initial

In [29]:
def jaccard_similarity(a, b):

    if not a or not b:
        return 0

    inter = len(a & b)
    union = len(a | b)

    return inter / union

In [30]:
def fuzzy_name_similarity(a, b):

    if not a or not b:
        return 0

    return fuzz.token_sort_ratio(a, b) / 100

In [31]:
def affiliation_similarity(a, b):

    if not a or not b:
        return 0

    s1 = " ".join(a)
    s2 = " ".join(b)

    fuzzy = fuzz.token_set_ratio(s1, s2) / 100
    jacc = jaccard_similarity(a, b)

    return (fuzzy + jacc) / 2

In [32]:
def compute_similarity_score(r1, r2):

    score = 0

    # nome
    name_sim = fuzzy_name_similarity(
        r1["normalized_name"],
        r2["normalized_name"]
    )
    score += name_sim * 3

    # discipline
    disc_sim = jaccard_similarity(
        r1["disciplines"],
        r2["disciplines"]
    )
    score += disc_sim * 1.5

    # affiliazione
    aff_sim = affiliation_similarity(
        r1["affiliation_tokens"],
        r2["affiliation_tokens"]
    )
    score += aff_sim * 2

    # consistenza temporale
    if r1["year"] and r2["year"]:

        gap = abs(r1["year"] - r2["year"])

        if gap <= 2:
            score += 1.5
        elif gap > 10:
            score -= 2

    return score

In [33]:
def normalize_similarity(score):

    if score < 0:
        score = 0

    return score / (score + 1)

In [34]:
def build_distance_matrix(records):

    n = len(records)

    distance_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i + 1, n):

            sim = compute_similarity_score(records[i], records[j])
            sim = normalize_similarity(sim)

            dist = 1 - sim

            distance_matrix[i][j] = dist
            distance_matrix[j][i] = dist

    return distance_matrix

In [42]:
def find_best_threshold(distance_matrix):

    thresholds = np.linspace(0.1, 0.9, 17)

    best_score = -1
    best_threshold = None
    best_labels = None

    n_samples = len(distance_matrix)

    for t in thresholds:

        clustering = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=t
        )

        labels = clustering.fit_predict(distance_matrix)

        n_clusters = len(set(labels))

        # condizioni valide per silhouette
        if n_clusters < 2 or n_clusters >= n_samples:
            continue

        score = silhouette_score(
            distance_matrix,
            labels,
            metric="precomputed"
        )

        if score > best_score:

            best_score = score
            best_threshold = t
            best_labels = labels

    return best_threshold, best_score, best_labels

In [36]:
def cluster_coherence(records, labels):

    clusters = defaultdict(list)

    for r, l in zip(records, labels):
        clusters[l].append(r)

    scores = []

    for c in clusters.values():

        if len(c) < 2:
            continue

        sims = []

        for i in range(len(c)):
            for j in range(i + 1, len(c)):

                s = compute_similarity_score(c[i], c[j])
                sims.append(s)

        scores.append(np.mean(sims))

    return np.mean(scores) if scores else 0

In [40]:
def disambiguate_authors(author_dict):

    revised_dict = {}

    for author_id, author_block in author_dict.items():

        records = []

        for author_name, info in author_block.items():

            affiliations_by_year = info.get("affiliations_by_year", {})
            papers = info.get("papers", {})

            for paper_title, paper_info in papers.items():

                year = paper_info.get("year")

                disciplines = set(
                    paper_info.get("macroareas", [])
                )

                affiliations = affiliations_by_year.get(year, [])

                collaborators = [
                    (c["name"], c["id"])
                    for c in paper_info.get("collaborators", [])
                ]

                records.append({

                    "paper": paper_title,
                    "name": author_name,
                    "normalized_name": normalize_author_name(author_name),
                    "year": year,
                    "disciplines": disciplines,
                    "affiliation_tokens": normalize_affiliations(affiliations),
                    "coauthors": collaborators
                })

        if len(records) < 2:

            revised_dict[author_id] = author_block
            continue


        ####################################################
        # BLOCKING
        ####################################################

        blocks = defaultdict(list)

        for r in records:
            key = blocking_key(r["name"])
            blocks[key].append(r)


        ####################################################
        # CLUSTERING PER BLOCCO
        ####################################################

        for block_records in blocks.values():

            if len(block_records) < 2:
                continue

            distance_matrix = build_distance_matrix(block_records)

            threshold, sil_score, labels = find_best_threshold(distance_matrix)

            if labels is None:
                continue

            coherence = cluster_coherence(block_records, labels)

            print(
                f"{author_id} block size {len(block_records)} | "
                f"threshold {threshold:.2f} | "
                f"silhouette {sil_score:.3f} | "
                f"coherence {coherence:.3f}"
            )

            unique_clusters = set(labels)

            for cluster_label in unique_clusters:

                new_id = f"{author_id}_cluster_{cluster_label}"

                cluster_records = [
                    block_records[i]
                    for i in range(len(block_records))
                    if labels[i] == cluster_label
                ]

                cluster_structure = {}

                for r in cluster_records:

                    name = r["name"]

                    if name not in cluster_structure:

                        cluster_structure[name] = {
                            "affiliations_by_year": {},
                            "papers": {}
                        }

                    original_info = author_block[name]

                    cluster_structure[name]["papers"][r["paper"]] = \
                        original_info["papers"][r["paper"]]

                    year = r["year"]

                    if year in original_info["affiliations_by_year"]:

                        cluster_structure[name]["affiliations_by_year"][year] = \
                            original_info["affiliations_by_year"][year]

                revised_dict[new_id] = cluster_structure

    return revised_dict

In [49]:
import re
import numpy as np
from difflib import SequenceMatcher
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score


# ─────────────────────────────────────────────
# UTILITY
# ─────────────────────────────────────────────

def normalize_name(name: str) -> str:
    """Normalizza solo spazi multipli e case, senza rimuovere caratteri."""
    return re.sub(r"\s+", " ", name.lower().strip())


def strong_name_mismatch(n1: str, n2: str) -> bool:
    """
    Hard constraint: se i cognomi sono chiaramente diversi → distanza massima.
    Normalizza solo spazi e case, senza rimuovere caratteri.
    """
    surname1 = normalize_name(n1).split()[-1]
    surname2 = normalize_name(n2).split()[-1]
    return surname1 != surname2


# ─────────────────────────────────────────────
# SIMILARITY
# ─────────────────────────────────────────────

def compute_similarity_score(r1: dict, r2: dict) -> float:
    score = 0.0

    # 1. Similarità nome (SequenceMatcher come proxy Levenshtein)
    name_sim = SequenceMatcher(
        None,
        normalize_name(r1["name"]),
        normalize_name(r2["name"])
    ).ratio()
    score += name_sim * 3

    # 2. Coautori con ID condivisi (segnale fortissimo)
    ids_1 = {cid for _, cid in r1["coauthors"]}
    ids_2 = {cid for _, cid in r2["coauthors"]}
    shared_ids = ids_1 & ids_2
    score += len(shared_ids) * 4

    # 3. Discipline condivise
    shared_disc = r1["disciplines"] & r2["disciplines"]
    score += len(shared_disc) * 0.5

    # 4. Similarità affiliazione
    if r1["affiliation"] and r2["affiliation"]:
        aff_sim = SequenceMatcher(
            None,
            r1["affiliation"].lower(),
            r2["affiliation"].lower()
        ).ratio()
        score += aff_sim * 2

    # 5. Consistenza temporale
    if r1["year"] and r2["year"]:
        year_gap = abs(r1["year"] - r2["year"])
        if year_gap <= 2:
            score += 1.5
        elif year_gap > 10:
            score -= 2

    return score


# ─────────────────────────────────────────────
# METRICHE DI VALUTAZIONE
# ─────────────────────────────────────────────

def compute_silhouette(distance_matrix: np.ndarray, labels: np.ndarray):
    """
    Silhouette Score su matrice di distanze precomputed.
    Restituisce None se il clustering ha un solo cluster o meno di 3 sample.
    Range: [-1, +1]. Più alto = cluster più compatti e separati.
    """
    n_labels = len(set(labels))
    if n_labels < 2 or len(labels) < 3:
        return None
    try:
        return silhouette_score(distance_matrix, labels, metric="precomputed")
    except Exception:
        return None


def compute_coherence_score(
    records: list,
    labels: np.ndarray,
    distance_matrix: np.ndarray
) -> dict:
    """
    Coherence Score personalizzato per la disambiguazione autori.

    Per ogni cluster calcola:
      - intra_dist_mean : distanza media intra-cluster (bassa = coeso)
      - inter_dist_min  : distanza minima verso il cluster più vicino (alta = separato)
      - coherence       : inter_dist_min / (intra_dist_mean + 1e-9)

    Score globale = media pesata per dimensione cluster.
    """
    unique_labels = sorted(set(labels))
    n = len(records)
    cluster_stats = {}

    for cl in unique_labels:
        idx_in  = [i for i in range(n) if labels[i] == cl]
        idx_out = [i for i in range(n) if labels[i] != cl]

        if len(idx_in) > 1:
            intra_dists = [
                distance_matrix[i][j]
                for ii, i in enumerate(idx_in)
                for j in idx_in[ii + 1:]
            ]
            intra_mean = float(np.mean(intra_dists))
        else:
            intra_mean = 0.0

        if idx_out:
            inter_min = float(np.min(distance_matrix[np.ix_(idx_in, idx_out)]))
        else:
            inter_min = 1.0

        coherence = inter_min / (intra_mean + 1e-9)

        cluster_stats[cl] = {
            "size": len(idx_in),
            "intra_dist_mean": round(intra_mean, 4),
            "inter_dist_min": round(inter_min, 4),
            "coherence": round(coherence, 4),
        }

    total = sum(s["size"] for s in cluster_stats.values())
    global_coherence = sum(
        s["coherence"] * s["size"] for s in cluster_stats.values()
    ) / (total or 1)

    return {
        "clusters": cluster_stats,
        "global_coherence": round(global_coherence, 4),
    }


# ─────────────────────────────────────────────
# DISAMBIGUAZIONE PRINCIPALE
# ─────────────────────────────────────────────

def disambiguate_authors(author_dict: dict) -> tuple:
    """
    Disambiguazione autori con valutazione Silhouette + Coherence.

    Restituisce:
        revised_dict : dizionario autori disambiguati (stesso formato input)
        metrics      : {author_id: {"silhouette": ..., "coherence": {...}}}
    """
    revised_dict = {}
    metrics = {}

    for author_id, author_block in author_dict.items():

        records = []

        for author_name, info in author_block.items():
            affiliations_by_year = info.get("affiliations_by_year", {})
            papers = info.get("papers", {})

            for paper_title, paper_info in papers.items():
                year         = paper_info.get("year")
                disciplines  = set(paper_info.get("macroareas", []))
                affiliations = affiliations_by_year.get(year, [])
                collaborators = [
                    (c["name"], c["id"])
                    for c in paper_info.get("collaborators", [])
                    if c.get("id") is not None
                ]

                records.append({
                    "paper":       paper_title,
                    "name":        author_name,
                    "year":        year,
                    "affiliation": " ".join(affiliations),
                    "disciplines": disciplines,
                    "coauthors":   collaborators,
                })

        n = len(records)

        if n < 2:
            revised_dict[author_id] = author_block
            metrics[author_id] = {"silhouette": None, "coherence": None,
                                   "note": "meno di 2 record"}
            continue

        # Matrice delle distanze
        distance_matrix = np.zeros((n, n))

        for i in range(n):
            for j in range(i + 1, n):
                if strong_name_mismatch(records[i]["name"], records[j]["name"]):
                    distance = 1.0
                else:
                    sim      = compute_similarity_score(records[i], records[j])
                    distance = 1 / (1 + max(sim, 0))

                distance_matrix[i][j] = distance
                distance_matrix[j][i] = distance

        # Clustering
        clustering = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=0.5,
        )
        labels = clustering.fit_predict(distance_matrix)
        unique_clusters = set(labels)

        # Valutazione
        sil = compute_silhouette(distance_matrix, labels)
        coh = compute_coherence_score(records, labels, distance_matrix)

        metrics[author_id] = {
            "n_records":        n,
            "n_clusters":       len(unique_clusters),
            "silhouette_score": round(sil, 4) if sil is not None else None,
            "coherence":        coh,
        }

        print(
            f"{author_id}: {len(unique_clusters)} cluster | "
            f"silhouette={metrics[author_id]['silhouette_score']} | "
            f"global_coherence={coh['global_coherence']}"
        )

        if len(unique_clusters) == 1:
            revised_dict[author_id] = author_block
            continue

        for cluster_label in unique_clusters:
            new_id = f"{author_id}_cluster_{cluster_label}"
            cluster_records = [records[i] for i in range(n) if labels[i] == cluster_label]

            cluster_structure = {}

            for r in cluster_records:
                name = r["name"]

                if name not in cluster_structure:
                    cluster_structure[name] = {
                        "affiliations_by_year": {},
                        "papers": {},
                    }

                original_info = author_block[name]
                cluster_structure[name]["papers"][r["paper"]] = \
                    original_info["papers"][r["paper"]]

                year = r["year"]
                if year in original_info["affiliations_by_year"]:
                    cluster_structure[name]["affiliations_by_year"][year] = \
                        original_info["affiliations_by_year"][year]

            revised_dict[new_id] = cluster_structure

    return revised_dict, metrics


# ─────────────────────────────────────────────
# REPORT AGGREGATO (opzionale)
# ─────────────────────────────────────────────

def print_global_report(metrics: dict) -> None:
    """Stampa un report aggregato su tutti gli autori valutati."""
    sil_scores = [
        m["silhouette_score"]
        for m in metrics.values()
        if m.get("silhouette_score") is not None
    ]
    coh_scores = [
        m["coherence"]["global_coherence"]
        for m in metrics.values()
        if m.get("coherence") is not None
    ]

    print("\n" + "=" * 50)
    print("REPORT GLOBALE DISAMBIGUAZIONE")
    print("=" * 50)
    print(f"Autori valutati         : {len(metrics)}")
    print(f"Con silhouette score    : {len(sil_scores)}")
    if sil_scores:
        print(f"  Media silhouette      : {np.mean(sil_scores):.4f}")
        print(f"  Min  silhouette       : {np.min(sil_scores):.4f}")
        print(f"  Max  silhouette       : {np.max(sil_scores):.4f}")
    if coh_scores:
        print(f"  Media coherence       : {np.mean(coh_scores):.4f}")
    print("=" * 50)

In [51]:
# Attuazione della name disambiguation
revised_dict, metrics = disambiguate_authors(author_dict)

https://openalex.org/A5010972450: 5 cluster | silhouette=0.8281 | global_coherence=20000006.3548
https://openalex.org/A5081230172: 1 cluster | silhouette=None | global_coherence=6.958
https://openalex.org/A5047187911: 2 cluster | silhouette=0.8121 | global_coherence=5.3223
https://openalex.org/A5068840916: 1 cluster | silhouette=None | global_coherence=17.0688
https://openalex.org/A5084828559: 1 cluster | silhouette=None | global_coherence=3.9339
https://openalex.org/A5100626378: 3 cluster | silhouette=0.9527 | global_coherence=22.2132
https://openalex.org/A5056306774: 1 cluster | silhouette=None | global_coherence=2.6383
https://openalex.org/A5000517206: 1 cluster | silhouette=None | global_coherence=15.3327
https://openalex.org/A5078790563: 1 cluster | silhouette=None | global_coherence=7.2841
https://openalex.org/A5080269874: 2 cluster | silhouette=0.8815 | global_coherence=8.584
https://openalex.org/A5024199819: 2 cluster | silhouette=0.8377 | global_coherence=6.1715
https://openal

In [52]:
print_global_report(metrics)


REPORT GLOBALE DISAMBIGUAZIONE
Autori valutati         : 86153
Con silhouette score    : 2860
  Media silhouette      : 0.8033
  Min  silhouette       : 0.3243
  Max  silhouette       : 0.9926
  Media coherence       : 10932921.0535


In [53]:
revised_dict

{'https://openalex.org/A5010972450_cluster_0': {'Valter Torri': {'affiliations_by_year': {2009: ['Mario Negri Institute for Pharmacological Research'],
    2007: ['Mario Negri Institute for Pharmacological Research'],
    2012: ['Mario Negri Institute for Pharmacological Research'],
    2005: ['European Institute of Oncology',
     'University of Milano-Bicocca',
     'University of Turin'],
    2016: ['Mario Negri Institute for Pharmacological Research',
     'Humanitas University'],
    2014: ['Mario Negri Institute for Pharmacological Research'],
    2019: ['Istituti di Ricovero e Cura a Carattere Scientifico',
     'Mario Negri Institute for Pharmacological Research'],
    2011: ['Mario Negri Institute for Pharmacological Research'],
    2015: ['Mario Negri Institute for Pharmacological Research'],
    2021: ['Ospedale San Giovanni Antica Sede'],
    2020: ['Mario Negri Institute for Pharmacological Research'],
    2018: ['Istituti di Ricovero e Cura a Carattere Scientifico',
     

In [57]:
def modify_structure(dictionary):
    new_dictionary = {}

    for id, author_block in dictionary.items():

        for name, info in author_block.items():
            new_dictionary[name] = {}

            affiliations_by_year = info.get('affiliations_by_year', {})
            papers = info.get('papers', {})

            ### PAPERS BLOCK ###
            collaborators_by_paper = {}

            for paper, paper_info in papers.items():
                collaboratori = []
                for collaborator in paper_info['collaborators']:
                    collaboratori.append(collaborator['name'])

                collaborators_by_paper[paper] = {
                    'collaborators': collaboratori,
                    'macroareas': paper_info['macroareas'],
                    'year': paper_info.get('year')
                }

            new_dictionary[name]['collaborators_by_paper'] = collaborators_by_paper

            ### ANNO BLOCK ###
            new_dictionary[name]['anno'] = affiliations_by_year

    return new_dictionary

In [58]:
new_dictionary = modify_structure(revised_dict)

In [60]:
with open('new_merged_authors.json', 'w') as outfile:
    json.dump(new_dictionary, outfile)